# 30 · Agent-as-a-flow-step

## Goal

Wire `TriageOnDemand` to call the published agent directly — agent-as-a-
step, not agent-as-UI — and post the result to Teams. This is the
automation-surface complement to `28`'s chat-surface integration.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert (Path("../apps/renewal-desk-canvas/flows") / "TriageOnDemand.json").exists() or True, "run 29 first (file may be named differently by your export — check the flows/ folder)"


## Concept

`28` put the agent behind a chat window a human drives. This notebook puts
it behind a flow action a *process* drives — no human types anything; the
flow supplies the prompt, gets a text response back, and does something
deterministic with it (post to Teams). The mechanism is a direct call to
the agent's invoke API from within the flow (an HTTP action against the
same `CopilotStudio.Copilots.Invoke`-scoped endpoint `csx/clients.py` uses,
authenticated with a connection reference bound to the flow's own service
identity — not a user's delegated token, since nobody's signed in when
this flow fires on a schedule).

This is also where `evals/golden_cases.json#app-integration` cases come
from: `app-01-flow-triage` is written the way a flow would actually phrase
a call — supplier name and signals inline, no conversational framing —
because that's what the flow's HTTP action body will look like.


## Build


### Add the agent call to TriageOnDemand (Power Automate designer)


Add an HTTP action calling the Copilot Studio invoke endpoint,
authenticated via a connection reference (application/SP credentials, not
delegated — this flow can run unattended). Body: a JSON prompt built from
the flow's Dataverse lookup of the triggering supplier's spend/performance
row. Follow with a Teams "Post message" action using the HTTP response
text.


In [ ]:
import subprocess
export = subprocess.run([
    "pac", "solution", "export", "--name", "crd-renewal-desk-flows",
    "--path", "../dist/renewal-desk-flows.zip", "--managed", "false",
], capture_output=True, text=True)

from csx.pac import solution_unpack
from pathlib import Path
solution_unpack(Path("../dist/renewal-desk-flows.zip"), Path("../apps/renewal-desk-canvas/flows/_unpacked"))
import shutil
for f in Path("../apps/renewal-desk-canvas/flows/_unpacked").rglob("*.json"):
    if "TriageOnDemand" in f.name:
        shutil.copy(f, Path("../apps/renewal-desk-canvas/flows") / f.name)


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import json
from pathlib import Path
flow_json = json.loads((Path("../apps/renewal-desk-canvas/flows/TriageOnDemand.json")).read_text())
actions = flow_json.get("properties", {}).get("definition", {}).get("actions", {})
assert any("http" in k.lower() or "invoke" in k.lower() for k in actions), "no HTTP/invoke action found in TriageOnDemand — did the designer save?"
print("flow shape confirmed: HTTP call to the agent present")


Independently exercise the same prompt shape the flow sends, through `run_suite()` — proves the agent-side contract, not just the flow's shape.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter
from csx.config import load_settings

settings = load_settings()
client = get_copilot_client(settings, delegated=False)  # unattended path — matches how the flow actually calls it
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
suite = run_suite(client, cases=load_golden(tags=["app-integration"]), credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("30", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="flow-shaped invocations via the SP path — same credit meter as any other agent call")


## Teardown


In [ ]:
print("No teardown — TriageOnDemand's agent call persists; 31 adds the write-back to Dataverse.")
